# M2 · Segmentación de instancias → tipo de vehículo
**IAM-811 · Laboratorio 3 · Sistema de control de peaje**

Clases: `moto`, `auto`, `minibus`, `camion`, `bus`. Cada vehículo recibe su máscara y su categoría, que define la tarifa base.

**Dataset** (Roboflow, proyecto de tipo *Instance Segmentation*):
1. Partir de un dataset de Roboflow Universe de vehículos con máscaras (opcional) y **ampliarlo con fotos propias** (carpetas `moto`, `bus`, `camion`, `autos_de_costado`).
2. Etiquetar polígonos con *Smart Polygon* (SAM) y revisar. Renombrar clases a las 5 de arriba.
3. Generar versión: Resize 640, aumentaciones: flip horizontal, brillo ±20 %, desenfoque 1 px, ruido 1 % (x3).
4. Exportar en formato **YOLOv11** y anotar aquí WORKSPACE / PROYECTO / VERSIÓN.

Entorno de ejecución: **GPU T4** (Entorno de ejecución → Cambiar tipo).

In [ ]:
# Versiones fijadas (mismas que inferencia/requirements.txt)
!pip install -q ultralytics==8.3.240 roboflow==1.1.66

In [ ]:
# Diagnóstico del entorno (punto adicional: gestión de entornos). Guardar la salida en la carpeta de evidencias.
import sys, platform, subprocess
import torch, ultralytics
print("python:", sys.version.split()[0])
print("sistema:", platform.platform())
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("ultralytics:", ultralytics.__version__)
subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv", shell=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
import os; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
# La API key se lee de los "Secretos" de Colab (icono de llave a la izquierda): nombre ROBOFLOW_API_KEY.
# NUNCA la escribas en el notebook ni la subas al repositorio.
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))

WORKSPACE = 'tu-workspace'      # <-- cambiar
PROYECTO = 'peaje-vehiculos-seg'  # <-- cambiar
VERSION = 1                      # <-- cambiar
dataset = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11')
print(dataset.location)

In [ ]:
# Conteo de imágenes e instancias por clase (para el informe).
import yaml, glob, collections, os
cfg = yaml.safe_load(open(f'{dataset.location}/data.yaml'))
nombres = cfg['names']
for parte in ['train', 'valid', 'test']:
    etiquetas = glob.glob(f'{dataset.location}/{parte}/labels/*.txt')
    conteo = collections.Counter()
    for f in etiquetas:
        for linea in open(f):
            if linea.strip():
                conteo[nombres[int(linea.split()[0])]] += 1
    print(f'{parte}: {len(etiquetas)} imágenes', dict(conteo))

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n-seg.pt')   # transfer learning desde COCO
resultados = modelo.train(
    data=f'{dataset.location}/data.yaml', epochs=80, imgsz=640, batch=16, patience=20,
    project='/content/runs', name='m2_seg', seed=42, cos_lr=True,
)

In [ ]:
# Métricas en validación y test (mAP de máscaras y cajas) para el informe.
mejor = YOLO('/content/runs/m2_seg/weights/best.pt')
for parte in ['val', 'test']:
    try:
        m = mejor.val(data=f'{dataset.location}/data.yaml', split=parte, imgsz=640)
        print(f'[{parte}] máscara mAP50={m.seg.map50:.3f} mAP50-95={m.seg.map:.3f} | caja mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}')
        for i, n in enumerate(m.names.values()):
            print(f'   {n:8s} mask mAP50-95={m.seg.maps[i]:.3f}')
    except Exception as e:
        print(parte, 'no disponible:', e)

In [ ]:
# Guardar pesos y gráficas en Drive (evidencias). Copiar m2_seg.pt a inferencia/modelos/ en RunPod.
import shutil
shutil.copy('/content/runs/m2_seg/weights/best.pt', f'{SALIDA}/m2_seg.pt')
shutil.copytree('/content/runs/m2_seg', f'{SALIDA}/m2_seg_run', dirs_exist_ok=True)
print('Listo:', os.listdir(SALIDA))

In [ ]:
# Prueba visual
from IPython.display import Image, display
r = mejor.predict(source=f'{dataset.location}/test/images', save=True, max_det=10, conf=0.35, project='/content/runs', name='m2_pred')
import glob
for f in sorted(glob.glob('/content/runs/m2_pred/*.jpg'))[:4]:
    display(Image(f, width=480))